In [1]:
%matplotlib inline
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# Classes with the physics engine
class SchwarzschildSpacetime:
    def __init__(self, M):
        self.M = M

    def null_geodesic(self, lmbda, y, b):
        t, r, phi, pr = y
        if r <= 2.001 * self.M:
            return [0, 0, 0, 0]

        dt_dl = 1.0 / (1.0 - 2.0 * self.M / r)
        dr_dl = pr
        dphi_dl = b / (r**2)
        dpr_dl = (b**2 / r**3) - (3.0 * self.M * b**2 / r**4)

        return [dt_dl, dr_dl, dphi_dl, dpr_dl]

    def trace_ray(self, source, b, direction='inward', lmbda_max=100, points=1000):
        try:
            y0 = source.get_initial_conditions(self.M, b, direction)
        except ValueError:
            return None # Skip rays that are physically impossible

        def hit_horizon(lmbda, y, b_arg):
            return y[1] - 2.001 * self.M
        hit_horizon.terminal = True

        eval_points = np.linspace(0, lmbda_max, points)

        solution = solve_ivp(
            self.null_geodesic, (0, lmbda_max), y0, args=(b,),
            method='RK45', t_eval=eval_points, events=hit_horizon,
            rtol=1e-8, atol=1e-8
        )
        return solution

class LightSource:
    def __init__(self, r0, phi0, label="Lamp"):
        self.r0 = r0
        self.phi0 = phi0
        self.label = label

    def get_initial_conditions(self, M, b, direction):
        V_eff = (1.0 - 2.0 * M / self.r0) * (b**2 / self.r0**2)
        if V_eff > 1.0:
            raise ValueError(f"Impact parameter b={b} too large.")
        pr0 = np.sqrt(1.0 - V_eff)
        return [0.0, self.r0, self.phi0, -pr0 if direction == 'inward' else pr0]

In [ ]:
# Simulation
def interactive_simulation(r1, phi1_deg, r2, phi2_deg):
    M = 1.0
    spacetime = SchwarzschildSpacetime(M)

    # Convert degrees from sliders into radians
    phi1 = np.radians(phi1_deg)
    phi2 = np.radians(phi2_deg)

    sources = [
        LightSource(r0=r1 * M, phi0=phi1, label="Lamp 1"),
        LightSource(r0=r2 * M, phi0=phi2, label="Lamp 2")
    ]

    b_values = np.linspace(0, 15, 1000)*M

    plt.figure(figsize=(8, 8))
    colors = ['orange', 'cyan']

    for i, source in enumerate(sources):
        color = colors[i]

        # Plot lamp position
        x0 = source.r0 * np.cos(source.phi0)
        y0 = source.r0 * np.sin(source.phi0)
        plt.plot(x0, y0, '*', color=color, markersize=15, label=source.label)

        # Plot rays
        for b in b_values:
            sol = spacetime.trace_ray(source, b, direction='inward')
            if sol is not None: # Only plot if the initial conditions were valid
                r_vals = sol.y[1]
                phi_vals = sol.y[2]
                x = r_vals * np.cos(phi_vals)
                y_cart = r_vals * np.sin(phi_vals)

                ray_label = f"Rays from {source.label}" if b == b_values[0] else None
                plt.plot(x, y_cart, color=color, alpha=0.7, label=ray_label)

    # Draw Black Hole Background
    horizon = plt.Circle((0,0), 2*M, color='black', label='Event Horizon')
    photon_sphere = plt.Circle((0,0), 3*M, color='red', fill=False, linestyle=':', label='Photon Sphere')
    plt.gca().add_patch(horizon)
    plt.gca().add_patch(photon_sphere)

    plt.xlabel('x (M)')
    plt.ylabel('y (M)')
    plt.title('Interactive Schwarzschild Ray Tracing')
    plt.legend(loc='upper right')
    plt.axis('equal')

    # Lock the axis limits so the plot doesn't jump around when moving sliders
    plt.xlim(-15, 15)
    plt.ylim(-15, 15)
    plt.grid(True, alpha=0.3)
    plt.show()

# Create the interactive UI sliders
widgets.interact(
    interactive_simulation,
    r1=widgets.FloatSlider(value=10.0, min=4.0, max=14.0, step=0.5, description='Lamp 1 R:'),
    phi1_deg=widgets.IntSlider(value=0, min=0, max=360, step=15, description='Lamp 1 Ang:'),
    r2=widgets.FloatSlider(value=10.0, min=4.0, max=14.0, step=0.5, description='Lamp 2 R:'),
    phi2_deg=widgets.IntSlider(value=90, min=0, max=360, step=15, description='Lamp 2 Ang:')
);

interactive(children=(FloatSlider(value=10.0, description='Lamp 1 R:', max=14.0, min=4.0, step=0.5), IntSlider…